In [1]:
import json
import os

import pandas as pd

from doc_chat.rag.multi_tenant_vector_store import MultiTenantVectorStore, \
    VectorStoreDocument

In [2]:
# set the number of files to process, None means all files
number_of_articles = 10
file_path = '../../data/squad/raw/train-v1.1.json'

results_directory = '../../data/squad/results'
if not os.path.exists(results_directory):
    os.makedirs(results_directory)

In [3]:
chroma_persist_directory = '/Users/patrick/projects/doc-chat/app_data/chroma_eval'


class Split:
    def __init__(self, content, page=0):
        self.page_content = content
        self.metadata = {"page": page}


if not os.path.exists(chroma_persist_directory):
    os.makedirs(chroma_persist_directory)
user_id = 'squad_semantic_search_text_embedding_3_large'
vector_store = MultiTenantVectorStore(chroma_persist_directory=chroma_persist_directory,
                                      embedding_model='text-embedding-3-large')

Using Chroma persist directory: /Users/patrick/projects/doc-chat/app_data/chroma_eval


In [4]:
# index the articles
with open(file_path, 'r') as f:
    squad_data = json.load(f)

print('Number of articles:', len(squad_data['data']))
articles = squad_data['data']

for article_idx, article in enumerate(articles):
    if number_of_articles and article_idx >= number_of_articles:
        break
    title = article['title']
    paragraphs = article['paragraphs']
    document_splits = []
    for p_idx, paragraph in enumerate(paragraphs):
        context = paragraph['context']
        document_splits.append(Split(content=context, page=p_idx))

    print(f"Indexing {len(document_splits)} chunks for article '{title}'")
    collection_id = f"article_{article_idx}"

    if vector_store.collection_exists(user_id=user_id, collection_id=collection_id):
        print(f"Collection '{collection_id}' already exists, deleting it.")
        vector_store.delete_collection(user_id=user_id, collection_id=collection_id)

    vector_store.create_document_collection(user_id=user_id,
                                            collection_id=collection_id,
                                            document_splits=document_splits,
                                            file_name=title)

collections = vector_store.get_collections(user_id=user_id)
print(collections)

Number of articles: 442
Indexing 55 chunks for article 'University_of_Notre_Dame'
Indexing 66 chunks for article 'Beyoncé'
Indexing 44 chunks for article 'Montana'
Indexing 26 chunks for article 'Genocide'
Indexing 26 chunks for article 'Antibiotics'
Indexing 82 chunks for article 'Frédéric_Chopin'
Indexing 72 chunks for article 'Sino-Tibetan_relations_during_the_Ming_dynasty'
Indexing 60 chunks for article 'IPod'
Indexing 32 chunks for article 'The_Legend_of_Zelda:_Twilight_Princess'
Indexing 43 chunks for article 'Spectre_(2015_film)'
['article_9', 'article_4', 'article_0', 'article_8', 'article_3', 'article_5', 'article_6', 'article_7', 'article_2', 'article_1']


In [5]:
# query the vector store

top_k = 5  # number of documents to retrieve


def mean_reciprocal_rank(retrieved_docs, doc_id):
    for doc_idx, doc in enumerate(retrieved_docs):
        if doc.id == doc_id:
            return 1 / (doc_idx + 1)
    return 0.0


def recall_at_k(retrieved_docs, doc_id):
    return 1 if any(doc.id == doc_id for doc in retrieved_docs) else 0


with open(file_path, 'r') as f:
    squad_data = json.load(f)

articles = squad_data['data']
data = []  # to store the results

for article_idx, article in enumerate(articles):
    if number_of_articles and article_idx >= number_of_articles:
        break
    title = article['title']
    collection_id = f"article_{article_idx}"
    paragraphs = article['paragraphs']

    print(f"Processing article '{title}' with {len(paragraphs)} paragraphs and collection ID '{collection_id}'")

    for p_idx, paragraph in enumerate(paragraphs):
        qas = paragraph['qas']
        for qa in qas:
            question = qa['question']
            question_id = qa['id']
            expected_doc_id = f"doc_{p_idx}"

            #print(f"Querying for question: '{question}' in article '{title}'")
            documents: list[VectorStoreDocument] = vector_store.retrieve_documents(user_id=user_id,
                                                                                   collection_id=collection_id,
                                                                                   query=question, k=top_k)
            # How high the correct doc ranks in top-k
            mrr = mean_reciprocal_rank(documents, expected_doc_id)

            # Whether the correct doc is found in top-k
            recall = recall_at_k(documents, expected_doc_id)

            data.append({
                'question_id': question_id,
                'article': title,
                'expected_doc_id': expected_doc_id,
                'retrieved_docs': [doc.id for doc in documents],
                'mrr': mrr,
                'recall': recall,
                'top_k': top_k,
                'number_of_paragraphs': len(paragraphs),
            })
            # print(f"Expected: {expected_doc_id}, found: {[doc.id for doc in documents]}")
            # print(f"Recall: {recall}, Precision: {mrr:.2f}")
            # print()

results_df = pd.DataFrame(data)
results_df.to_csv(f'{results_directory}/results_top_{top_k}_large_embed.csv', index=False)

Processing article 'University_of_Notre_Dame' with 55 paragraphs
Processing article 'Beyoncé' with 66 paragraphs
Processing article 'Montana' with 44 paragraphs
Processing article 'Genocide' with 26 paragraphs
Processing article 'Antibiotics' with 26 paragraphs
Processing article 'Frédéric_Chopin' with 82 paragraphs
Processing article 'Sino-Tibetan_relations_during_the_Ming_dynasty' with 72 paragraphs
Processing article 'IPod' with 60 paragraphs
Processing article 'The_Legend_of_Zelda:_Twilight_Princess' with 32 paragraphs
Processing article 'Spectre_(2015_film)' with 43 paragraphs


In [8]:
# read the results
def print_results(data_set):
    results_df = pd.read_csv(f'{results_directory}/{data_set}')
    print(data_set)
    print(f"Number of questions: {len(results_df)}")
    print(f"Number of articles: {results_df['article'].nunique()}")
    mean_mrr = results_df['mrr'].mean()
    mean_recall = results_df['recall'].mean()
    print(f"Mean MRR: {mean_mrr:.4f}")
    print(f"Mean Recall: {mean_recall:.4f}")
    print()


print_results("results_top_5_large_embed.csv")
print_results("results_top_5_small_embed.csv")

results_top_5_large_embed.csv
Number of questions: 3286
Number of articles: 10
Mean MRR: 0.8096
Mean Recall: 0.9391

results_top_5_small_embed.csv
Number of questions: 3286
Number of articles: 10
Mean MRR: 0.7675
Mean Recall: 0.9130

